<a href="https://colab.research.google.com/github/gliao6578/super_mariners/blob/main/generate_ais_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import csv
import random
from datetime import datetime, timedelta
from google.colab import files

# This script added weather anomaly into the list.

# 1. Configuration & Setup
# @markdown ### Set Dataset Generation Size
NUM_RECORDS = 100000  # @param {type:"integer"}

# Enter percentage of anomalies
ANOMALY_PERCENT = .25  # @param {type:"number"}
NUM_ANOMALIES = int(NUM_RECORDS * (ANOMALY_PERCENT / 100))
NUM_NORMAL = NUM_RECORDS - NUM_ANOMALIES

# Base Fleet Data (Normal Profiles)
vessels = [
    {"mmsi": 366999001, "imo": 9999001, "name": "PACIFIC VOYAGER", "call": "WXYZ1", "type": 70, "draft": 11.2, "dest": "LOS ANGELES"},
    {"mmsi": 367111222, "imo": 9111222, "name": "SAN DIEGO SPIRIT", "call": "ABCD2", "type": 80, "draft": 8.5, "dest": "LONG BEACH"},
    {"mmsi": 368333444, "imo": 9333444, "name": "HARBOR TUG ONE", "call": "EFGH3", "type": 52, "draft": 4.1, "dest": "SAN DIEGO"},
    {"mmsi": 477123400, "imo": 9123400, "name": "OCEAN GIANT", "call": "VRHA5", "type": 70, "draft": 14.5, "dest": "SHANGHAI"},
    {"mmsi": 211234560, "imo": 9234560, "name": "NORDIC STAR", "call": "OUJW2", "type": 80, "draft": 12.1, "dest": "ROTTERDAM"},
    {"mmsi": 235111000, "imo": 9111000, "name": "ATLANTIC TRADER", "call": "2EBC4", "type": 71, "draft": 10.8, "dest": "SINGAPORE"},
    {"mmsi": 311000123, "imo": 9000123, "name": "SEA BREEZE", "call": "C6BI8", "type": 60, "draft": 6.2, "dest": "VANCOUVER"},
    {"mmsi": 412000456, "imo": 9000456, "name": "ORIENT EXPRESS", "call": "BOQA9", "type": 70, "draft": 13.0, "dest": "TOKYO"},
    {"mmsi": 563000789, "imo": 9000789, "name": "GLOBAL CARRIER", "call": "9VBC1", "type": 70, "draft": 15.2, "dest": "BUSAN"},
    {"mmsi": 636000111, "imo": 9000111, "name": "CAPE TRADER", "call": "D5RN2", "type": 70, "draft": 11.5, "dest": "PANAMA CANAL"}
]

anomaly_types = [
    "Navigational Status Conflict",
    "Missing Heading (511 Error)",
    "Geographic Spoofing / Outliers",
    "Impossible Speed Spike",
    "MMSI Spoofing (Invalid Identifier)",
    "Frozen/Stale Telemetry",
    "Extreme Course Drifting",
    "Destination/ETA Logical Conflict",
    "Invalid/Corrupt Data Types (Negative)",
    "Ghost Vessel (Zeroed Out IMO/Name)",
    "Weather-Impacted Navigation Anomaly"
]

start_time = datetime(2026, 10, 9, 12, 0, 0)
rows = []

# 2. Generate Normal Traffic (Normal - Anomaly = Normal Traffic)
for i in range(NUM_NORMAL):
    v = random.choice(vessels)
    tstamp = start_time + timedelta(seconds=i * 2)
    eta = tstamp + timedelta(days=random.randint(2, 10))

    cog = round(random.uniform(0.0, 359.9), 1)
    heading = int(cog)

    rows.append([
        v["mmsi"], tstamp.strftime("%Y-%m-%dT%H:%M:%SZ"),
        round(random.uniform(32.0, 33.5), 4), round(random.uniform(-118.5, -116.5), 4),
        cog, round(random.uniform(0.5, 22.0), 1), heading, 0,
        v["imo"], v["name"], v["call"], v["type"], v["draft"], v["dest"],
        eta.strftime("%Y-%m-%dT%H:%M:%SZ"), 0, "Normal"
    ])

# 3. Generate Anomalous Traffic (distributed evenly, handling remainder)
anomalies_per_type = NUM_ANOMALIES // len(anomaly_types)
remainder = NUM_ANOMALIES % len(anomaly_types)

for a_idx, a_type in enumerate(anomaly_types):
    # Distribute the remaining anomalies across the first few categories
    count_to_generate = anomalies_per_type + (1 if a_idx < remainder else 0)
    for k in range(count_to_generate):
        v = random.choice(vessels)
        tstamp = start_time + timedelta(seconds=random.randint(0, 50000))
        eta = tstamp + timedelta(days=random.randint(2, 10))
        cog = round(random.uniform(0.0, 359.9), 1)

        # Base Defaults
        mmsi, imo, name, call, v_type, draft = v["mmsi"], v["imo"], v["name"], v["call"], v["type"], v["draft"]
        lat, lon = round(random.uniform(32.0, 33.5), 4), round(random.uniform(-118.5, -116.5), 4)
        sog = round(random.uniform(5.0, 20.0), 1)
        heading = int(cog)
        navstat = 0

        # Apply distinct mutations based on anomaly rules
        if a_idx == 0:  # Navigational Status Conflict
            navstat = random.choice([1, 5]) # Anchored or Moored
            sog = round(random.uniform(18.5, 26.0), 1)
        elif a_idx == 1:  # Missing Heading
            heading = 511
            sog = round(random.uniform(15.0, 24.0), 1)
        elif a_idx == 2:  # Geographic Spoofing
            lat, lon = round(random.uniform(51.0, 55.0), 4), round(random.uniform(2.0, 5.0), 4) # North Sea
        elif a_idx == 3:  # Impossible Speed Spike
            sog = round(random.uniform(65.0, 150.0), 1)
        elif a_idx == 4:  # MMSI Spoofing
            mmsi = random.choice([0, 12345, 999999999999])
        elif a_idx == 5:  # Frozen Telemetry
            lat, lon, cog, heading = 32.5555, -117.5555, 90.0, 90
        elif a_idx == 6:  # Extreme Course Drifting
            heading = (int(cog) + 210) % 360
        elif a_idx == 7:  # Temporal Logical Conflict
            eta = tstamp - timedelta(days=random.randint(1, 5)) # ETA in the past
        elif a_idx == 8:  # Invalid/Corrupt Data Types
            draft, sog = -9.9, -2.5
        elif a_idx == 9:  # Ghost Vessel
            imo, name, call = 0, "UNKNOWN", "XXXX"
        elif a_idx == 10: # Weather-Impacted Navigation Anomaly
            sog = round(random.uniform(0.1, 1.5), 1) # Barely making steerageway due to heavy weather
            heading = (int(cog) + random.choice([45, -45, 90, -90])) % 360 # Severe wind/wave shearing forces

        rows.append([
            mmsi, tstamp.strftime("%Y-%m-%dT%H:%M:%SZ"), lat, lon,
            cog, sog, heading, navstat, imo, name, call, v_type, draft, v["dest"],
            eta.strftime("%Y-%m-%dT%H:%M:%SZ"), 1, a_type
        ])

# Shuffle to mix anomalies evenly with normal rows
random.shuffle(rows)

# 4. Save and Download File
headers = ["MMSI", "TSTAMP", "LATITUDE", "LONGITUDE", "COG", "SOG", "HEADING", "NAVSTAT", "IMO", "NAME", "CALLSIGN", "TYPE", "DRAUGHT", "DEST", "ETA", "IS_ANOMALY", "ANOMALY_TYPE"]
filename = f"ais_shipping_traffic_{len(rows)}_complex_anomalies.csv"

with open(filename, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(headers)
    writer.writerows(rows)

print(f"Successfully generated {filename} with {len(rows)} records!")

try:
    files.download(filename)
    print("Download triggered automatically!")
except Exception as e:
    print(f"Download could not be triggered automatically: {e}")

Successfully generated ais_shipping_traffic_100000_complex_anomalies.csv with 100000 records!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download triggered automatically!


# New Section